# 1000 Genomes PCA — Population Structure Analysis

**Chromosome 1 · bcftools + PLINK 1.9 + Python**

This notebook documents the full pipeline used to detect population (ancestry) structure
from real human genetic data using Principal Component Analysis (PCA). It was built as
part of a self-training assignment on GWAS fundamentals.

**What this shows:** starting from ~6.47 million raw genetic variants on chromosome 1,
cleaning and filtering down to 17,840 high-quality markers, and running PCA to see
whether 2,504 individuals cluster by continental ancestry — purely from unlabeled
genotype data.

---

**Environment:** WSL2 (Ubuntu 24.04) · bcftools 1.19 · PLINK 1.9 · Python (pandas, matplotlib)


## 1. Pipeline Overview

| Step | Tool | Purpose |
|---|---|---|
| 1. Download | `wget` | Get chr1 VCF, reference genome, ancestry panel, strict mask |
| 2. Normalize | `bcftools norm` | Standardize variant representation, remove duplicates |
| 3. Convert | `plink --make-bed` | VCF/BCF → PLINK binary format |
| 4. Mask | `plink --extract` | Keep only high-confidence genomic regions |
| 5. QC + Prune | `plink --maf`, `--indep-pairwise` | Common SNPs only, remove redundant markers |
| 6. PCA | `plink --pca` | Compress genetic variation into principal components |
| 7. Plot | `pandas` + `matplotlib` | Visualize PC1 vs PC2, colored by ancestry |

Each step is explained below, including the commands used and any problems encountered.


## 2. Step 1 — Download the Data

Four files were downloaded for chromosome 1:
- **VCF**: `ALL.chr1.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz` — the genetic variant data
- **Reference genome**: `human_g1k_v37.fasta` — the baseline human sequence variants are compared against
- **Ancestry panel**: `integrated_call_samples_v3.20130502.ALL.panel` — each person's known population/super-population label
- **Strict mask**: `20141020.strict_mask.whole_genome.bed` — marks genomic regions considered reliably sequenced

> **Problem faced:** Windows PowerShell's `wget` is actually `Invoke-WebRequest`, which doesn't handle
> large file downloads reliably, and Windows has no `gunzip`. **Fix:** switched to WSL (Ubuntu), which has
> real Linux tools. Large downloads (900MB–1.1GB) also kept timing out mid-transfer on an unstable connection —
> resolved using `wget -c` to resume rather than restart, and verifying every file with `gzip -t` before use.


In [ ]:
# Download commands (run in WSL shell, not this notebook)
# wget -c https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr1.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz
# wget https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/human_g1k_v37.fasta.gz
# wget https://ftp-trace.ncbi.nih.gov/1000genomes/ftp/release/20130502/integrated_call_samples_v3.20130502.ALL.panel
# wget https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/supporting/accessible_genome_masks/20141020.strict_mask.whole_genome.bed
print("See markdown above — commands run in the shell, not executed here.")

## 3. Step 2 — Normalize Variants (bcftools)

The same genetic variant can be written multiple different ways in a VCF file. **Normalization**
rewrites every variant in one standard, unambiguous form, following two principles:

1. **Parsimony** — describe the variant with the fewest letters possible (trim shared, non-informative bases)
2. **Left-alignment** — if a variant could sit at more than one equally valid position (common in repetitive
   DNA), always place it as far left as possible, so every tool agrees on its location

This step also removes duplicate entries.


In [ ]:
# bcftools norm -m-any --check-ref w -f human_g1k_v37.fasta \
#   ALL.chr1.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz | \
#   bcftools annotate -I +'%CHROM:%POS:%REF:%ALT' | \
#   bcftools norm -Ob --rm-dup both \
#     > ALL.chr1...genotypes.bcf
# bcftools index ALL.chr1...genotypes.bcf

# Result: 6,471,910 clean, standardized variants
print("Normalized variant count: 6,471,910")

## 4. Step 3 — Convert to PLINK Format

PLINK (the main tool used for filtering, QC, and PCA) doesn't read VCF/BCF directly for these
operations — it needs its own binary format (`.bed`/`.bim`/`.fam`). This is a pure format
conversion; no genetic data is changed.

> **Problem faced:** `sudo apt install plink` installed the wrong package (`putty-tools`, due to a
> naming collision in Ubuntu's repositories). **Fix:** installed the correct package directly —
> `sudo apt install plink1.9`.


In [ ]:
# plink --bcf ALL.chr1...genotypes.bcf --keep-allele-order --vcf-idspace-to _ \
#   --const-fid --allow-extra-chr 0 --split-x b37 no-fail --make-bed \
#   --out ALL.chr1...genotypes
print("2,504 people, 6,471,910 variants loaded successfully")

## 5. Step 4 — Apply the Strict Mask

Not every part of the genome can be sequenced reliably — some regions are too repetitive or
ambiguous for confident variant calling. The **strict mask** marks which regions passed strict
reliability criteria. Filtering to this mask reduces false-positive variant calls.

> **Problem faced:** the 123MB mask file kept losing its connection partway through the download,
> several times. **Fix:** `wget` automatically retried and resumed each time until the file
> completed and passed an integrity check.

**Result:** 6,471,910 → **4,851,384 variants**


In [ ]:
# plink --bfile ALL.chr1...genotypes --make-set 20141020.strict_mask.whole_genome.bed \
#   --write-set --out strict_mask_variants
# plink --bfile ALL.chr1...genotypes --extract strict_mask_variants.set \
#   --make-bed --out chr1_masked
print("Variants after masking: 4,851,384")

## 6. Step 5 — Quality Control (QC) and Pruning

Two sub-steps:

1. **QC**: keep only SNPs (excluding insertions/deletions) with **MAF (Minor Allele Frequency) > 10%**
   — common, well-established variants only. Rare variants appear in too few people to reveal
   ancestry patterns, so they add noise without much PCA signal.
2. **Pruning**: remove SNPs highly correlated with their neighbors (Linkage Disequilibrium / LD),
   so PCA isn't biased toward genomic regions with lots of tightly-linked markers.

**Result:** 4,851,384 → 308,967 variants (QC) → **17,840 variants** (after pruning)


In [ ]:
# plink --bfile chr1_masked --snps-only --maf 0.1 --make-bed --out chr1_qc
# plink --bfile chr1_qc --indep-pairwise 50 5 0.2 --out chr1_pruned
# plink --bfile chr1_qc --extract chr1_pruned.prune.in --make-bed --out chr1_final
print("Final PCA input: 17,840 variants, 2,504 individuals")

## 7. Step 6 — Run PCA

PCA compresses each person's 17,840 genetic markers down into a small number of **principal
components (PCs)** — numbers that capture the biggest directions of variation across everyone.

**Result:** PC1 explained **51.9%** of all variation, PC2 explained **24.1%** — together
capturing ~76% of the variation in just two numbers per person.


In [ ]:
# plink --bfile chr1_final --pca 20 --out chr1_pca
print("PCA complete: chr1_pca.eigenvec, chr1_pca.eigenval written")

## 8. Step 7 — Load Results and Plot

This section is real, executable code. It loads the PLINK PCA output, merges it with each
person's known ancestry label, and plots PC1 vs PC2.

> **Problem faced:** Ubuntu 24.04 blocks system-wide `pip install` with an
> `externally-managed-environment` error. **Fix:** created an isolated Python virtual
> environment (`python3 -m venv pcavenv`) and installed `pandas`/`matplotlib` inside it.

**Note:** to re-run this cell yourself, place `chr1_pca.eigenvec` and
`integrated_call_samples_v3.20130502.ALL.panel` in the same directory as this notebook,
or update the paths below.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load PCA results (no header; columns: FID, IID, PC1, PC2, ..., PC20)
pca_cols = ["FID", "IID"] + [f"PC{i}" for i in range(1, 21)]
pca = pd.read_csv("../results/pca/chr1_pca.eigenvec", sep=r"\s+", header=None, names=pca_cols)

# Load ancestry panel (columns: sample, pop, super_pop, gender)
panel = pd.read_csv("../results/pca/integrated_call_samples_v3.20130502.ALL.panel", sep="\t")

# Merge on sample ID
merged = pca.merge(panel, left_on="IID", right_on="sample", how="left")

fig, ax = plt.subplots(figsize=(9, 7))
groups = merged.groupby("super_pop")
colors = plt.cm.tab10.colors

for (name, group), color in zip(groups, colors):
    ax.scatter(group["PC1"], group["PC2"], label=name, alpha=0.6, s=15, color=color)

ax.set_xlabel("PC1 (51.9% variance explained)")
ax.set_ylabel("PC2 (24.1% variance explained)")
ax.set_title("1000 Genomes Chr1 PCA — colored by super population")
ax.legend(title="Super Population")
plt.tight_layout()
plt.savefig("../results/pca/chr1_pca_plot.png", dpi=150)
plt.show()

![PCA plot](../results/pca/chr1_pca_plot.png)

### Interpretation

- **East Asian (EAS)** and **European (EUR)** form the tightest, most clearly separated
  clusters — both descend from a relatively small number of ancestral founders.
- **African (AFR)** is spread out in a long streak rather than a tight blob — reflecting the
  real biological fact that human genetic diversity is highest within Africa, since our
  species originated there.
- **South Asian (SAS)** forms a moderately tight cluster between European and the center.
- **Admixed American (AMR)** is the most scattered group, stretching between the European,
  South Asian, and African clusters — consistent with real admixture history (European,
  African, and Indigenous American ancestry) in these populations.

**Caveat:** this used only chromosome 1, not the whole genome. PC1/PC2 explaining ~76% of
variance is unusually high; a typical whole-genome PCA on this dataset usually shows PC1
explaining only 5–10%. The clustering pattern is real and expected, but the strength of
separation is partly a side effect of using a smaller, more concentrated signal.


## 9. Minor Allele Frequency (MAF) Distribution

Before filtering to common variants, the overall frequency distribution across all
4,851,384 masked variants on chromosome 1 was examined using `plink --freq`.


In [ ]:
freq = pd.read_csv("../results/maf/chr1_freq.frq", sep=r"\s+")

common = (freq["MAF"] > 0.05).sum()
low_freq = ((freq["MAF"] >= 0.01) & (freq["MAF"] <= 0.05)).sum()
rare = (freq["MAF"] < 0.01).sum()
total = len(freq)

print(f"Total variants: {total}")
print(f"Common (MAF>5%):      {common} ({100*common/total:.2f}%)")
print(f"Low-frequency (1-5%): {low_freq} ({100*low_freq/total:.2f}%)")
print(f"Rare (MAF<1%):        {rare} ({100*rare/total:.2f}%)")

plt.figure(figsize=(8,5))
plt.hist(freq["MAF"], bins=50, edgecolor="black")
plt.xlabel("Minor Allele Frequency (MAF)")
plt.ylabel("Number of variants")
plt.title("MAF distribution — Chromosome 1 (1000 Genomes, strict-masked)")
plt.tight_layout()
plt.savefig("../results/maf/chr1_maf_distribution.png", dpi=150)
plt.show()

![MAF distribution](../results/maf/chr1_maf_distribution.png)

**Results:** Rare (MAF<1%): **83.47%** · Low-frequency (1–5%): **7.05%** · Common (MAF>5%): **9.47%**

Most genetic variation is rare and evolutionarily recent — mutations haven't had enough
generations to spread widely. Common variants are older and have had more time to drift
or be selected to higher frequency, which is why there are fewer of them but each is shared
by far more people. This is also why PCA used MAF > 10% — rare variants are numerous but
individually uninformative for capturing broad ancestry structure.


## 10. Key Takeaways

- Population structure is visible directly from DNA, with no ancestry labels given to the
  analysis — PCA recovered continental ancestry groupings purely from unlabeled genotypes.
- Data cleaning is as important as the final analysis: normalization, masking, QC, and
  pruning each shaped the result.
- Real analysis involves real friction — broken downloads, wrong package names, environment
  restrictions — working through these is a normal part of the process.

## Next Steps

- Association analysis (GWAS logistic/linear regression, Manhattan plot, QQ plot) — see `02_GWAS_Association.ipynb`
- Post-GWAS analysis: LD Score Regression (LDSC) for heritability/confounding, and variant annotation
